# ECG Arrhythmia Classification: Sequence Models vs Tabular Baseline
- **Author:** Jose García Mayén
- **Date:** 07/2026
- **Dataset:** MIT-BIH Arrhythmia Database (48 records, 360 Hz)

## Project objective
Classify every heartbeat into **3 AAMI classes** on the honest **inter-patient**
benchmark (de Chazal DS1/DS2 — no patient shared between train and test):

| Label | Class | Share of test (DS2) |
|-------|-------|---------------------|
| 0 (N) | Normal + LBBB + RBBB + junctional | ~89% |
| 1 (S) | Supraventricular ectopic | ~3.7% |
| 2 (V) | Ventricular ectopic + Fusion | ~7.3% |

Paced beats (`Q`/`/`) are dropped and Fusion (`F`) is merged into V, per AAMI.

We compare three models:
- **XGBoost** — tabular baseline on 187 raw samples + 46 engineered features (per beat, no cross-beat context).
- **Transformer (Many-to-Many)** — self-attention over a window of `W=45` beats.
- **Seq2Seq BiLSTM (Many-to-Many)** — recurrence over the same 45-beat window.

**Target:** `f1_macro ≥ 0.80`. Best so far ≈ **0.758** (Transformer / Seq2Seq).
The bottleneck is class **S**, which is defined by *rhythm* (RR intervals), not beat
morphology — hence the sequence models that see neighbouring beats.


## How to run this notebook

By default this notebook is **exploratory and fast**: it loads the precomputed
datasets and the pretrained models from Kaggle Datasets and only *evaluates* them.

The heavy stages (raw data creation from WFDB, hyperparameter search, training) are
kept for reference but guarded by flags that are **OFF by default**:

```python
RUN_DATA_CREATION = False   # rebuild CSV/npy from raw MIT-BIH
RUN_TUNING        = False   # Optuna search
RUN_TRAINING      = False   # train from scratch
```

Attach the Kaggle Dataset **`heartwaveml-artifacts`** (single dataset with three folders).
If your slug differs, edit `PREP` / `MODELS` below:
- `feat/mitbih_{train,cv,test}_features.csv`
- `seq/{train,cv,test}_{X,y,y_seq}.npy`
- `models/{modelXGB.joblib, modelTransformer.keras, modelSeq2Seq.keras}`


## Setup
---

In [ ]:
# The pretrained models were saved with Keras 2 (TF 2.10). Kaggle ships Keras 3,
# which cannot read those files, so we install tf-keras and force the legacy Keras
# API (see the env var in the next cell). wfdb is only needed for data creation.
!pip install -q tf-keras wfdb

In [ ]:
import os

# Load the Keras-2 (.keras) pretrained models on Kaggle's Keras-3 image.
# Must be set BEFORE importing tensorflow.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import re
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, confusion_matrix, ConfusionMatrixDisplay,
)

import tensorflow as tf

plt.rcParams["figure.figsize"] = [11, 6]


def set_seed(seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)


set_seed(42)

In [ ]:
# ---- Paths: single Kaggle Dataset "heartwaveml-artifacts" with feat/ seq/ models/ ----
PREP   = "/kaggle/input/heartwaveml-artifacts"
MODELS = "/kaggle/input/heartwaveml-artifacts/models"

# ---- Run flags: heavy stages OFF by default (exploratory, fast) ----
RUN_DATA_CREATION = False   # rebuild CSV/npy from raw MIT-BIH (WFDB download + feature extraction)
RUN_TUNING        = False   # Optuna hyperparameter search
RUN_TRAINING      = False   # train models from scratch; default loads pretrained

WINDOW = 45
CENTER = WINDOW // 2                 # center beat index of each sequence
CLASS_NAMES = {0: "N", 1: "S", 2: "V"}
LABELS = [0, 1, 2]

# de Chazal inter-patient partition (patient-wise, no leakage)
DS1_RECORDS = {"101","106","108","109","112","114","115","116","118","119",
               "122","124","201","203","205","207","208","209","215","220","223","230"}
DS2_RECORDS = {"100","103","105","111","113","117","121","123","200","202","210",
               "212","213","214","219","221","222","228","231","232","233","234"}
CV_RECORDS    = {"223", "201", "118"}      # carved from DS1, rich in S to estimate F1-S
TRAIN_RECORDS = DS1_RECORDS - CV_RECORDS

# 1. Datasets creation (optional)
---

Everything in this section runs only when `RUN_DATA_CREATION = True`. On Kaggle you normally skip it and load the precomputed artifacts in section 2.

We use the **WFDB** package to read the raw ECG signals and the cardiologist
annotations (beat positions ≈ R-peaks, and AAMI beat symbols) from the PhysioNet
records. `record.p_signal[:, 0]` is the MLII lead at 360 Hz; `ann.sample` holds the
beat centers and `ann.symbol` the labels.

In [ ]:
if RUN_DATA_CREATION:
    import wfdb
    os.makedirs("mitbih", exist_ok=True)
    wfdb.dl_database("mitdb", dl_dir="mitbih")

    # Quick look at one record with its beat labels
    rec = wfdb.rdrecord("mitbih/100", sampto=360 * 5)
    ann = wfdb.rdann("mitbih/100", "atr", sampto=360 * 5)
    ecg = rec.p_signal[:, 0]
    t = np.arange(len(ecg)) / rec.fs
    plt.plot(t, ecg, "b", linewidth=1)
    for s, sym in zip(ann.sample, ann.symbol):
        plt.text(s / rec.fs + 0.1, ecg[s], sym, color="red")
    plt.xlabel("Time (s)"); plt.ylabel("Amplitude (mV)"); plt.title("ECG with beat labels")
    plt.show()

### Beat labels → 3 AAMI classes

We keep the AAMI grouping but reduce to **3 classes**. `class_mapping` assigns the
raw symbol to a coarse group; downstream we **drop paced/unknown (4)** and **merge
Fusion (3) into V (2)**, leaving N / S / V.

In [ ]:
class_mapping = {
    "N": 0, "·": 0, "L": 0, "R": 0, "e": 0, "j": 0,   # N: normal + bundle branch + junctional
    "A": 1, "a": 1, "J": 1, "S": 1,                    # S: supraventricular ectopic
    "V": 2, "E": 2,                                    # V: ventricular ectopic
    "F": 3,                                            # Fusion -> merged into V below
    "/": 4, "f": 4, "x": 4, "Q": 4, "|": 4, "~": 4,    # paced / unknown -> dropped below
}

`generateData` segments a 187-sample window centered on each R-peak and records the
**`record` id** and **`beat_center`** sample (needed later for the RR/HRV features and
for the patient-wise split).

In [ ]:
def generateData(record_numbers, window_size=187):
    rows = []
    for rec_id in record_numbers:
        record = wfdb.rdrecord(f"mitbih/{rec_id}")
        ann = wfdb.rdann(f"mitbih/{rec_id}", "atr")
        sig = record.p_signal[:, 0]
        w2 = window_size // 2
        for i in range(len(ann.sample)):
            symbol = ann.symbol[i]
            if symbol in class_mapping:
                center = ann.sample[i]
                start = max(0, center - w2)
                end = min(len(sig), center + w2 + window_size % 2)
                if end - start == window_size:
                    row = sig[start:end].tolist() + [class_mapping[symbol], rec_id, center]
                    rows.append(row)
    cols = [f"sample_{i}" for i in range(window_size)] + ["class", "record", "beat_center"]
    return pd.DataFrame(rows, columns=cols)


if RUN_DATA_CREATION:
    import wfdb
    record_numbers = sorted(
        m.group(1) for f in os.listdir("mitbih")
        if (m := re.match(r"^(\d+)\.atr$", f))
    )
    combined = generateData(record_numbers)
    combined.to_csv("mitbih_combined_records.csv", index=False)
    print("combined:", combined.shape)
    print(combined["class"].value_counts().sort_index())

## Feature engineering
---

Per beat we compute **36 morphological features** (statistics, amplitude, energy,
crossings, peaks and 5 segment summaries) plus **10 HRV / RR-interval features**
(computed per record from `beat_center`). The RR block is what carries the *rhythm*
information that separates class S. **Invariant: the 10 RR features are always the
last 10 columns.**

In [ ]:
def extract_features_from_beat(beat_signal, fs=360):
    features = {}
    if len(beat_signal) < 10 or np.all(beat_signal == 0):
        return features
    peaks = find_peaks(beat_signal, height=np.max(beat_signal) * 0.3, distance=int(0.2 * fs))
    features["mean"] = np.mean(beat_signal)
    features["std"] = np.std(beat_signal)
    features["var"] = np.var(beat_signal)
    features["median"] = np.median(beat_signal)
    features["mad"] = np.median(np.abs(beat_signal - np.median(beat_signal)))
    features["skewness"] = skew(beat_signal)
    features["kurtosis"] = kurtosis(beat_signal)
    features["max_val"] = np.max(beat_signal)
    features["min_val"] = np.min(beat_signal)
    features["range"] = np.max(beat_signal) - np.min(beat_signal)
    features["peak_to_peak"] = np.ptp(beat_signal)
    features["energy"] = np.sum(beat_signal ** 2)
    features["power"] = np.mean(beat_signal ** 2)
    features["rms"] = np.sqrt(np.mean(beat_signal ** 2))
    features["zero_crossings"] = len(np.where(np.diff(np.signbit(beat_signal)))[0])
    features["mean_crossings"] = len(np.where(np.diff(np.signbit(beat_signal - np.mean(beat_signal))))[0])
    features["r_peak_std_ratio"] = features["max_val"] / (features["var"] ** 0.5 + 1e-6)
    features["num_peaks"] = len(peaks)
    features["r_peak_amplitude"] = np.max(beat_signal)
    features["r_peak_position"] = np.argmax(beat_signal) / len(beat_signal)
    features["total_area"] = np.trapz(np.abs(beat_signal))
    n_segments = 5
    seg_len = len(beat_signal) // n_segments
    for i in range(n_segments):
        s = i * seg_len
        e = (i + 1) * seg_len if i < n_segments - 1 else len(beat_signal)
        segment = beat_signal[s:e]
        features[f"segment_{i}_mean"] = np.mean(segment)
        features[f"segment_{i}_std"] = np.std(segment)
        features[f"segment_{i}_area"] = np.trapz(np.abs(segment))
    return features


RR_COLS = ["rr_pre", "rr_post", "rr_ratio", "rr_local_mean", "rr_global_mean",
           "rr_pre_norm_local", "rr_post_norm_local", "rr_pre_norm_global",
           "rr_local_std", "rr_rmssd"]


def compute_rr_features(df):
    rr = pd.DataFrame(index=df.index, columns=RR_COLS, dtype=float)
    for _, group in df.groupby("record", sort=False):
        centers = group["beat_center"].values
        n = len(centers)
        pre, post = np.empty(n), np.empty(n)
        if n > 1:
            intervals = (centers[1:] - centers[:-1]) / 360.0
            pre[1:] = intervals; post[:-1] = intervals
            pre[0] = pre[1]; post[-1] = post[-2]
        else:
            pre[0] = post[0] = 0.0
        ratio = pre / (post + 1e-6)
        s = pd.Series(pre)
        local_mean = s.rolling(10, min_periods=1).mean().to_numpy()
        global_mean = s.expanding(min_periods=1).mean().to_numpy()
        local_std = s.rolling(10, min_periods=2).std().fillna(0.0).to_numpy()
        rmssd = s.diff().pow(2).rolling(5, min_periods=1).mean().apply(np.sqrt).fillna(0.0).to_numpy()
        idx = group.index
        rr.loc[idx, "rr_pre"] = pre
        rr.loc[idx, "rr_post"] = post
        rr.loc[idx, "rr_ratio"] = ratio
        rr.loc[idx, "rr_local_mean"] = local_mean
        rr.loc[idx, "rr_global_mean"] = global_mean
        rr.loc[idx, "rr_pre_norm_local"] = pre / (local_mean + 1e-6)
        rr.loc[idx, "rr_post_norm_local"] = post / (local_mean + 1e-6)
        rr.loc[idx, "rr_pre_norm_global"] = pre / (global_mean + 1e-6)
        rr.loc[idx, "rr_local_std"] = local_std
        rr.loc[idx, "rr_rmssd"] = rmssd
    return rr


def build_features(df):
    signal_cols = [c for c in df.columns if c.startswith("sample_")]
    feats = pd.DataFrame([extract_features_from_beat(df[signal_cols].iloc[i].values)
                          for i in range(len(df))])
    rr = compute_rr_features(df)
    return pd.concat([df[signal_cols].reset_index(drop=True),
                      feats.reset_index(drop=True),
                      rr.reset_index(drop=True),
                      df[["class", "record", "beat_center"]].reset_index(drop=True)], axis=1)


if RUN_DATA_CREATION:
    combined = pd.read_csv("mitbih_combined_records.csv")
    features_df = build_features(combined)
    features_df.to_csv("mitbih_features.csv", index=False)
    print("features:", features_df.shape)

## Inter-patient split, filtering and scaling
---

The split is **patient-wise** (de Chazal DS1/DS2): train and CV come from DS1, test is
DS2, so no patient appears in two sets — this is the honest inter-patient paradigm and
avoids leakage. **No SMOTE** (class imbalance is handled at train time via sample
weights). A 40 Hz Butterworth low-pass is applied to the **187 raw samples only** — the
46 engineered features are computed on the raw signal by design and must not be
filtered. The scaler is fit on **train only**.

In [ ]:
def get_filter_coeffs(cutoff_freq=40, fs=360, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff_freq / nyq, btype="low", analog=False)
    return b, a


def apply_filter(data, b, a):
    return filtfilt(b, a, data)


def split_and_scale(features_df):
    df = features_df.copy()
    df = df[df["class"] != 4].reset_index(drop=True)      # drop paced/unknown
    df["class"] = df["class"].replace({3: 2})              # Fusion -> V (AAMI)
    rec = df["record"].astype(str)
    drop_cols = [c for c in ["class", "record", "beat_center"] if c in df.columns]
    X = df.drop(drop_cols, axis=1)
    y = df["class"]
    sample_cols = [c for c in X.columns if c.startswith("sample_")]
    b, a = get_filter_coeffs()

    def filt(X_split):
        X_split = X_split.copy()
        X_split[sample_cols] = np.apply_along_axis(
            apply_filter, 1, X_split[sample_cols].values, b=b, a=a)
        return X_split

    masks = {"train": rec.isin(TRAIN_RECORDS), "cv": rec.isin(CV_RECORDS), "test": rec.isin(DS2_RECORDS)}
    scaler = StandardScaler().fit(filt(X[masks["train"]]))
    out = {}
    for name, m in masks.items():
        scaled = pd.DataFrame(scaler.transform(filt(X[m])), columns=X.columns)
        scaled["class"] = y[m].reset_index(drop=True)
        out[name] = scaled
    return out


if RUN_DATA_CREATION:
    features_df = pd.read_csv("mitbih_features.csv")
    feat_splits = split_and_scale(features_df)
    os.makedirs("feat", exist_ok=True)
    for name, d in feat_splits.items():
        d.to_csv(f"feat/mitbih_{name}_features.csv", index=False)
        print(name, d.shape)

## Sequence generation (for Transformer / Seq2Seq)
---

The sequence models see a window of `W=45` beats (22 before + center + 22 after).
Using only the **46 features** (no raw samples), we scale them (fit on train) and, **per
record**, build overlapping windows with edge clipping. For each center beat we store the
window `X (45,46)`, the center label `y`, and the per-position labels `y_seq (45,)` used
by the Many-to-Many loss.

In [ ]:
def generate_sequences(feat_only_df, window=WINDOW):
    K = window // 2
    df = feat_only_df.copy()
    df = df[df["class"] != 4].reset_index(drop=True)
    df["class"] = df["class"].replace({3: 2})
    feature_cols = [c for c in df.columns if c not in ("class", "record")]
    rec = df["record"].astype(str)
    masks = {"train": rec.isin(TRAIN_RECORDS).values,
             "cv": rec.isin(CV_RECORDS).values,
             "test": rec.isin(DS2_RECORDS).values}
    X_all = df[feature_cols].values.astype(np.float32)
    scaler = StandardScaler()
    X_all[masks["train"]] = scaler.fit_transform(X_all[masks["train"]])
    X_all[masks["cv"]] = scaler.transform(X_all[masks["cv"]])
    X_all[masks["test"]] = scaler.transform(X_all[masks["test"]])

    out = {}
    for name, m in masks.items():
        sx, sy, srec = X_all[m], df["class"].values[m], rec.values[m]
        Xs, ys, yseqs = [], [], []
        for r in pd.unique(srec):
            pos = np.where(srec == r)[0]
            rX, rY, n = sx[pos], sy[pos], len(pos)
            for i in range(n):
                idxs = np.clip(np.arange(i - K, i + K + 1), 0, n - 1)
                Xs.append(rX[idxs]); ys.append(rY[i]); yseqs.append(rY[idxs])
        out[name] = (np.array(Xs, np.float32), np.array(ys, np.int32), np.array(yseqs, np.int32))
        print(name, out[name][0].shape)
    return out, scaler


if RUN_DATA_CREATION:
    # features_only = the 46 feature columns + class + record (drop the raw samples)
    features_df = pd.read_csv("mitbih_features.csv")
    keep = [c for c in features_df.columns if not c.startswith("sample_") and c != "beat_center"]
    seq_out, seq_scaler = generate_sequences(features_df[keep])
    os.makedirs("seq", exist_ok=True)
    for name, (X, y, yseq) in seq_out.items():
        np.save(f"seq/{name}_X.npy", X); np.save(f"seq/{name}_y.npy", y); np.save(f"seq/{name}_y_seq.npy", yseq)
    joblib.dump(seq_scaler, "scaler_seq.joblib")

# 2. Load data & explore
---

Default fast path: load the precomputed feature CSVs and sequence arrays from the attached Kaggle Dataset.

In [ ]:
train_df = pd.read_csv(f"{PREP}/feat/mitbih_train_features.csv")
cv_df    = pd.read_csv(f"{PREP}/feat/mitbih_cv_features.csv")
test_df  = pd.read_csv(f"{PREP}/feat/mitbih_test_features.csv")

train_X_seq = np.load(f"{PREP}/seq/train_X.npy")
test_X_seq  = np.load(f"{PREP}/seq/test_X.npy")
test_y_seq  = np.load(f"{PREP}/seq/test_y.npy")

print("feat train/cv/test:", train_df.shape, cv_df.shape, test_df.shape)
print("seq train X:", train_X_seq.shape, " seq test X:", test_X_seq.shape)
feature_names = [c for c in train_df.columns if c != "class"]
print("n features:", len(feature_names), "(187 samples + 46 engineered)")
print("last 10 columns (RR, invariant):", feature_names[-10:])

### Class distribution per split

The strong imbalance (N ≫ V > S) is why we report **macro-F1** and weight the loss.

In [ ]:
splits = {"train": train_df, "cv": cv_df, "test": test_df}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (name, d) in zip(axes, splits.items()):
    counts = d["class"].value_counts().reindex(LABELS, fill_value=0)
    ax.bar([CLASS_NAMES[c] for c in LABELS], counts.values, color=["#4C72B0", "#DD8452", "#55A868"])
    ax.set_title(f"{name}  (n={len(d)})")
    for i, v in enumerate(counts.values):
        ax.text(i, v, f"{v}\n{100*v/len(d):.1f}%", ha="center", va="bottom", fontsize=9)
plt.suptitle("Class distribution N / S / V per split (inter-patient)")
plt.tight_layout(); plt.show()

### Mean raw beat per class

N and V separate well by *morphology*; the raw waveform alone barely distinguishes S from N.

In [ ]:
sample_cols = [c for c in train_df.columns if c.startswith("sample_")]
plt.figure(figsize=(11, 5))
for c in LABELS:
    beats = train_df.loc[train_df["class"] == c, sample_cols].values
    plt.plot(beats.mean(0), label=f"{CLASS_NAMES[c]}  (n={len(beats)})", linewidth=2)
plt.title("Mean heartbeat per class (filtered + scaled, 187 samples)")
plt.xlabel("Sample (360 Hz)"); plt.ylabel("Amplitude (scaled)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Why S needs rhythm, not morphology

Class S (supraventricular ectopic) is characterised by a **premature** beat — a short
`rr_pre` relative to the local rhythm. The RR feature `rr_pre_norm_local` (pre-interval
normalised by the rolling mean of the last 10) separates S far better than any
morphological feature, which is exactly why the sequence models help.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
data = [train_df.loc[train_df["class"] == c, "rr_pre_norm_local"].clip(-3, 3) for c in LABELS]
ax.boxplot(data, labels=[CLASS_NAMES[c] for c in LABELS], showfliers=False)
ax.set_title("rr_pre_norm_local by class (premature beat marker)")
ax.set_ylabel("rr_pre / rolling mean(10)  (scaled)")
ax.grid(alpha=0.3, axis="y"); plt.show()

### One input sequence (Transformer / Seq2Seq)

Each model input is a `(45, 46)` window: 45 consecutive beats × 46 features, centered on the beat to classify (index 22). `y` is the center label; `y_seq` holds all 45 labels for the Many-to-Many loss.

In [ ]:
w = test_X_seq[0]  # (45, 46)
plt.figure(figsize=(12, 5))
plt.imshow(w.T, aspect="auto", cmap="viridis")
plt.axvline(CENTER, color="red", ls="--", lw=1.5, label=f"center beat (idx {CENTER})")
plt.colorbar(label="scaled value")
plt.xlabel("beat position in window"); plt.ylabel("feature index (0-45)")
plt.title(f"One input window  shape={w.shape}  (center label = {CLASS_NAMES[test_y_seq[0]]})")
plt.legend(loc="upper right"); plt.show()

# 3. Models
---

Shared helpers, then each model: an **optional** training path (guarded) and the default **load-pretrained** path.

In [ ]:
def get_class_weights(y):
    y = pd.Series(y).reset_index(drop=True)
    classes = sorted(y.unique())
    n, k = len(y), len(classes)
    return {c: n / (k * (y == c).sum()) for c in classes}


def make_class_weight_array(y_center, shape_2d=None):
    # (N, W) sample weights for Many-to-Many: every position inherits the center-beat weight
    cw = get_class_weights(y_center)
    w = np.array([cw[int(c)] for c in y_center], dtype=np.float32)
    return w if shape_2d is None else np.tile(w[:, None], (1, shape_2d[1]))

## 3.1 XGBoost (tabular baseline)
---

Trained on `feat` (187 raw + 46 features) per beat, with balanced sample weights. It has no cross-beat context, so it is a strong-but-capped baseline.

In [ ]:
X_train_feat = train_df.drop("class", axis=1); y_train_feat = train_df["class"]
X_cv_feat    = cv_df.drop("class", axis=1);    y_cv_feat    = cv_df["class"]

if RUN_TUNING:
    import optuna
    from xgboost import XGBClassifier
    sw = pd.Series(y_train_feat).map(get_class_weights(y_train_feat)).values

    def objective(trial):
        params = dict(
            max_depth=trial.suggest_int("max_depth", 3, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            n_estimators=trial.suggest_int("n_estimators", 100, 500),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.0, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 0.0, 2.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            objective="multi:softprob", num_class=3, random_state=42,
        )
        m = XGBClassifier(**params)
        m.fit(X_train_feat, y_train_feat, sample_weight=sw, eval_set=[(X_cv_feat, y_cv_feat)], verbose=False)
        return f1_score(y_cv_feat, m.predict(X_cv_feat), average="macro")

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print("best:", study.best_trial.params)

In [ ]:
if RUN_TRAINING:
    from xgboost import XGBClassifier
    sw = pd.Series(y_train_feat).map(get_class_weights(y_train_feat)).values
    modelXGB = XGBClassifier(objective="multi:softprob", num_class=3,
                             eval_metric="mlogloss", n_jobs=-1, random_state=42)
    modelXGB.fit(X_train_feat, y_train_feat, sample_weight=sw,
                 eval_set=[(X_cv_feat, y_cv_feat)], verbose=False)
else:
    modelXGB = joblib.load(f"{MODELS}/modelXGB.joblib")

print("XGB ready:", type(modelXGB).__name__)

## 3.2 Transformer Encoder (Many-to-Many)
---

```
Input (W, 46) -> Dense(d_model) + sinusoidal positional encoding
             -> N x [MultiHeadAttention + Add&Norm + FFN + Add&Norm]
             -> TimeDistributed(Dense(3, linear))   -> (N, W, 3)
```

Loss = `SparseCategoricalCrossentropy(from_logits=True)` over **all** positions;
class imbalance handled with a `(N, W)` `sample_weight`. At inference we take
`argmax` at the **center** position only. Production HP: d_model=64, heads=4,
ff_dim=128, blocks=2, dropout=0.1, lr=1e-3.

In [ ]:
def positional_encoding(window, d_model):
    pos = np.arange(window)[:, None]
    dims = np.arange(d_model)[None, :]
    rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    ang = pos * rates
    ang[:, 0::2] = np.sin(ang[:, 0::2]); ang[:, 1::2] = np.cos(ang[:, 1::2])
    return tf.constant(ang[None, ...], dtype=tf.float32)


def transformer_block(x, num_heads, key_dim, ff_dim, dropout):
    from tensorflow.keras.layers import MultiHeadAttention, Add, LayerNormalization, Dense, Dropout
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    x = LayerNormalization()(Add()([x, attn]))
    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dropout(dropout)(ff)
    ff = Dense(x.shape[-1])(ff)
    return LayerNormalization()(Add()([x, ff]))


def build_transformer(window, n_features, d_model=64, num_heads=4, ff_dim=128,
                      num_blocks=2, dropout=0.1, learning_rate=1e-3):
    from tensorflow.keras.layers import Input, Dense, Dropout, TimeDistributed
    from tensorflow.keras.models import Model
    inp = Input(shape=(window, n_features))
    x = Dense(d_model)(inp)
    x = x + positional_encoding(window, d_model)
    for _ in range(num_blocks):
        x = transformer_block(x, num_heads, d_model // num_heads, ff_dim, dropout)
    x = TimeDistributed(Dense(32, activation="relu"))(x)
    x = TimeDistributed(Dropout(dropout))(x)
    out = TimeDistributed(Dense(3, activation="linear"))(x)
    model = Model(inp, out)
    model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  optimizer=tf.keras.optimizers.Adam(learning_rate),
                  metrics=["accuracy"], weighted_metrics=[])
    return model

In [ ]:
if RUN_TRAINING:
    y_train_c = np.load(f"{PREP}/seq/train_y.npy")
    y_train_s = np.load(f"{PREP}/seq/train_y_seq.npy")
    X_cv_s = np.load(f"{PREP}/seq/cv_X.npy"); y_cv_c = np.load(f"{PREP}/seq/cv_y.npy")
    y_cv_s = np.load(f"{PREP}/seq/cv_y_seq.npy")
    sw_tr = make_class_weight_array(y_train_c, y_train_s.shape)
    sw_cv = make_class_weight_array(y_cv_c, y_cv_s.shape)
    modelTransformer = build_transformer(WINDOW, train_X_seq.shape[2])
    modelTransformer.fit(train_X_seq, y_train_s,
                         validation_data=(X_cv_s, y_cv_s, sw_cv),
                         sample_weight=sw_tr, epochs=30, batch_size=256, verbose=1)
else:
    modelTransformer = tf.keras.models.load_model(f"{MODELS}/modelTransformer.keras", compile=False)

print("Transformer ready")

## 3.3 Seq2Seq BiLSTM (Many-to-Many)
---

```
Input (W, 46) -> Bidirectional(LSTM, return_sequences=True) x2 (+Dropout)
             -> TimeDistributed(Dense(dense_units, relu))
             -> TimeDistributed(Dense(3, linear))   -> (N, W, 3)
```

Same loss / `sample_weight (N, W)` / center-position inference as the Transformer.
A large window with per-position gradient is what lifts S recall.

In [ ]:
def build_seq2seq(window, n_features, units1=64, units2=32, dense_units=32,
                  dropout=0.3, learning_rate=1e-3):
    from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Dense, TimeDistributed
    from tensorflow.keras.models import Model
    inp = Input(shape=(window, n_features))
    x = Bidirectional(LSTM(units1, return_sequences=True))(inp)
    x = Dropout(dropout)(x)
    x = Bidirectional(LSTM(units2, return_sequences=True))(x)
    x = Dropout(dropout)(x)
    x = TimeDistributed(Dense(dense_units, activation="relu"))(x)
    x = TimeDistributed(Dropout(dropout))(x)
    out = TimeDistributed(Dense(3, activation="linear"))(x)
    model = Model(inp, out)
    model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  optimizer=tf.keras.optimizers.Adam(learning_rate),
                  metrics=["accuracy"], weighted_metrics=[])
    return model


if RUN_TRAINING:
    y_train_c = np.load(f"{PREP}/seq/train_y.npy")
    y_train_s = np.load(f"{PREP}/seq/train_y_seq.npy")
    X_cv_s = np.load(f"{PREP}/seq/cv_X.npy"); y_cv_s = np.load(f"{PREP}/seq/cv_y_seq.npy")
    y_cv_c = np.load(f"{PREP}/seq/cv_y.npy")
    sw_tr = make_class_weight_array(y_train_c, y_train_s.shape)
    sw_cv = make_class_weight_array(y_cv_c, y_cv_s.shape)
    modelSeq2Seq = build_seq2seq(WINDOW, train_X_seq.shape[2])
    modelSeq2Seq.fit(train_X_seq, y_train_s,
                     validation_data=(X_cv_s, y_cv_s, sw_cv),
                     sample_weight=sw_tr, epochs=30, batch_size=128, verbose=1)
else:
    modelSeq2Seq = tf.keras.models.load_model(f"{MODELS}/modelSeq2Seq.keras", compile=False)

print("Seq2Seq ready")

# 4. Evaluation (test = DS2)
---

Per-class F1 (N/S/V), macro-F1 and a normalized confusion matrix for each model, evaluated on the fully unseen inter-patient test set.

In [ ]:
def compute_metrics(y_true, y_pred):
    f1p = f1_score(y_true, y_pred, average=None, labels=LABELS, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_N": f1p[0], "f1_S": f1p[1], "f1_V": f1p[2],
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f2_macro": fbeta_score(y_true, y_pred, beta=2, average="macro", zero_division=0),
        "cm": confusion_matrix(y_true, y_pred, labels=LABELS, normalize="true"),
    }


def predict_tabular(model, df):
    return model.predict(df.drop("class", axis=1)), df["class"].values


def predict_seq(model, X):
    logits = model.predict(X, batch_size=512, verbose=0)   # (N, W, 3)
    return np.argmax(logits[:, X.shape[1] // 2, :], axis=1)


# XGB on feat test, sequence models on seq test
yp_xgb, y_true_tab = predict_tabular(modelXGB, test_df)
yp_tr  = predict_seq(modelTransformer, test_X_seq)
yp_s2s = predict_seq(modelSeq2Seq, test_X_seq)

results = {
    "XGBoost":     compute_metrics(y_true_tab, yp_xgb),
    "Transformer": compute_metrics(test_y_seq, yp_tr),
    "Seq2Seq":     compute_metrics(test_y_seq, yp_s2s),
}

In [ ]:
table = pd.DataFrame(
    {name: {k: m[k] for k in ["f1_N", "f1_S", "f1_V", "f1_macro", "f1_weighted", "accuracy"]}
     for name, m in results.items()}
).T
table = table.round(4).sort_values("f1_macro", ascending=False)
print("Test-set (DS2) metrics — target f1_macro >= 0.80")
display(table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, m) in zip(axes, results.items()):
    ConfusionMatrixDisplay(m["cm"], display_labels=[CLASS_NAMES[c] for c in LABELS]).plot(
        cmap=plt.cm.Blues, ax=ax, colorbar=False, values_format=".2f")
    ax.set_title(f"{name}\nmacro-F1 = {m['f1_macro']:.3f}")
plt.suptitle("Confusion matrices (row-normalized) — test set DS2")
plt.tight_layout(); plt.show()

In [ ]:
# Per-class F1 comparison — the S bar is the bottleneck vs the literature (~0.82)
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(3); width = 0.25
for i, (name, m) in enumerate(results.items()):
    ax.bar(x + (i - 1) * width, [m["f1_N"], m["f1_S"], m["f1_V"]], width, label=name)
ax.set_xticks(x); ax.set_xticklabels(["F1-N", "F1-S", "F1-V"])
ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.set_title("Per-class F1 by model (test DS2)")
ax.legend(); ax.grid(alpha=0.3, axis="y"); plt.show()

# 5. Conclusions & next steps
---

- The **sequence models** (Transformer, Seq2Seq) clearly beat the tabular XGBoost
  baseline on macro-F1, driven almost entirely by **class S** — seeing the 45-beat
  rhythm context lets them recover premature beats that a per-beat model cannot.
- We are still **below the target** `f1_macro ≥ 0.80` and below the literature on
  the same DS1/DS2 split (Farag 2023, Zahid 2022 report F1-S ≈ 0.82, macro ≈ 0.92).
  **S remains the bottleneck** (F1-S ≈ 0.43–0.52 here).

**What we can try next**
- **Larger window** (`W=101`): going 9 → 45 gave +0.19–0.23 macro; more context may help further (~2× slower).
- **3D FocalLoss** over `(N, W, 3)` to push S precision.
- **Re-tune the Transformer** now that sinusoidal positional encoding is in — current HP predate it.
- Validate on test the best Optuna config found in CV (not yet promoted).

*Reproduce the artifacts:* set `RUN_DATA_CREATION = True` to rebuild the CSV/npy from raw
MIT-BIH, and `RUN_TRAINING = True` to retrain the models.
